## Imports and Data

In [ ]:
# pip install msoffcrypto-tool

import io
import os
import msoffcrypto
import pandas as pd
import re

decrypted_workbook = io.BytesIO()
with open('data/Xapheresis.xlsx', 'rb') as file:
    office_file = msoffcrypto.OfficeFile(file)
    office_file.load_key(password=os.environ["XAPHERESIS_PASSWORD"])
    office_file.decrypt(decrypted_workbook)

# pandas
df = pd.read_excel(decrypted_workbook, skiprows=10)
print(f"Loaded {len(df)} administrations")

df.sample(20)

In [2]:
# All types of Generic

l = df["Slices by Component Simple Generic Name"].unique()
for _ in l:
    print(_)
print("Total:", len(l))

Albumin Human
KCentra-OR auto-verify
Balfaxar
Antithrombin III (Human) (Thrombate)
Antihemophilic Fac-VWF (Alphanate)
Antihemophil Fac (rAHF-PFM) (Advate)
Coag Fac VIIa Recomb (Novoseven)
Total: 7


In [3]:
# All types of Order Names

l = df["Order Name"].unique()
for _ in l:
    print(_)
print("Total:", len(l))

albumin human 25 % bottle 12.5 g
albumin human 5 % bottle 12.5 g
albumin human 5 % bottle  - ADS Override Pull
albumin human 25 % bottle 50 g
albumin human 5 % bottle 250 mL
albumin human 25 % bottle
albumin human 5 % bottle
albumin human 25 % bottle 25 g
albumin human 5 % bottle 25 g
albumin human 25 % bottle 73 g
albumin human 25 % bottle 22.5 g
albumin human 25 % bottle 15.25 g
albumin human 25 % bottle  - ADS Override Pull
albumin human 5 % bottle 6.15 g
albumin human 5 % bottle 61.5 mL
albumin human 25 % bottle 11 g
albumin human 5 % bottle 43 mL
albumin human 5 % bottle 4.3 g
albumin human 5 % bottle 2.55 g
albumin human 25 % bottle 70.25 g
albumin human 5 % bottle 50 g
four factor human prothrombin complex concentrate (Kcentra) injection
Prothrombin Complex Human-lans (Balfaxar) 2,000 Units 80 mL infusion
Prothrombin Complex Human-lans (Balfaxar) 1,000 Units 40 mL infusion
Prothrombin Complex Human-lans (Balfaxar) 1,500 Units 60 mL infusion
antithrombin III (human) (Thrombate II

## Drug Schema

Each entry defines how to identify and aggregate a drug subtype:
- **`patterns`**: list of regex patterns matched against `match_column` (any match = positive)
- **`match_column`**: column to run the regex against (defaults to `Order Name`)
- **`target_unit`**: the unit all amounts will be converted to
- **`unit_conversions`**: maps source unit → multiplier to reach `target_unit`; units not listed here are excluded
- **`dose_from_name_pattern`**: regex with two capture groups `(amount, unit)` to pull a dose from the Order Name when `Ordered Dose Amount` is missing or non-convertible
- **`default_dose`**: fallback `{"amount": ..., "unit": ...}` used when neither of the above resolves — set `amount` to `None` as a placeholder until a value is known

**Dose resolution priority per row:**
1. `Ordered Dose Amount` + `Ordered Dose Unit` (if present and unit is in `unit_conversions`)
2. Dose extracted from `Order Name` via `dose_from_name_pattern`
3. `default_dose["amount"]` + `default_dose["unit"]`

In [4]:
DRUG_SCHEMA = {
    # ── Albumin ──────────────────────────────────────────────────────────────
    "Albumin 5%": {
        "patterns": [r"albumin human 5 %"],
        "match_column": "Order Name",
        "target_unit": "g",
        "unit_conversions": {
            "g":  1.0,
            "mL": 0.05,   # 5% solution = 50 mg/mL = 0.05 g/mL
        },
        "dose_from_name_pattern": r"([\d\.]+)\s*(g|mL)(?:\s|$)",
        "default_dose": {"amount": 12.5, "unit": "g"},
    },
    "Albumin 25%": {
        "patterns": [r"albumin human 25 %"],
        "match_column": "Order Name",
        "target_unit": "g",
        "unit_conversions": {
            "g":  1.0,
            "mL": 0.25,   # 25% solution = 250 mg/mL = 0.25 g/mL
        },
        "dose_from_name_pattern": r"([\d\.]+)\s*(g|mL)(?:\s|$)",
        "default_dose": {"amount": 12.5, "unit": "g"},
    },

    # ── Prothrombin Complex Concentrates ─────────────────────────────────────
    "Balfaxar (4F-PCC)": {
        "patterns": [r"balfaxar", r"prothrombin complex human-lans"],
        "match_column": "Order Name",
        "target_unit": "Units",
        "unit_conversions": {
            "Units": 1.0,
        },
        # dose appears before volume, e.g. "1,000 Units 40 mL infusion"
        "dose_from_name_pattern": r"([\d,]+)\s*(Units)",
        "default_dose": {"amount": None, "unit": "Units"},  # TODO: fill in
    },
    "Kcentra (4F-PCC)": {
        "patterns": [r"kcentra"],
        "match_column": "Order Name",
        "target_unit": "Units",
        "unit_conversions": {
            "Units": 1.0,
        },
        "dose_from_name_pattern": r"([\d,]+)\s*(Units)",
        "default_dose": {"amount": None, "unit": "Units"},  # TODO: fill in
    },

    # ── Antithrombin ─────────────────────────────────────────────────────────
    "Antithrombin III (Thrombate)": {
        "patterns": [r"antithrombin"],
        "match_column": "Order Name",
        "target_unit": "Units",
        "unit_conversions": {
            "Units": 1.0,
        },
        "dose_from_name_pattern": r"([\d,]+)\s*(Units)\s*$",
        "default_dose": {"amount": None, "unit": "Units"},  # TODO: fill in
    },

    # ── Antihemophilic Factors ───────────────────────────────────────────────
    "Antihemophilic Factor-VWF (Alphanate)": {
        "patterns": [r"alphanate"],
        "match_column": "Order Name",
        "target_unit": "Units",
        # VWF:RCo Units and Int'l Units are both treated as units of drug activity
        "unit_conversions": {
            "VWF:RCo Units": 1.0,
            "Int'l Units":   1.0,
            "Units":         1.0,
        },
        "dose_from_name_pattern": r"([\d,]+)\s*(VWF:RCo Units|Int'l Units|Units)\s*$",
        "default_dose": {"amount": None, "unit": "Units"},  # TODO: fill in
    },
    "Antihemophilic Factor rAHF-PFM (Advate)": {
        "patterns": [r"rahf-pfm", r"advate"],
        "match_column": "Order Name",
        "target_unit": "Units",
        "unit_conversions": {
            "Units": 1.0,
        },
        "dose_from_name_pattern": r"([\d,]+)\s*(Units)\s*$",
        "default_dose": {"amount": None, "unit": "Units"},  # TODO: fill in
    },

    # ── Coagulation Factors ──────────────────────────────────────────────────
    "Coagulation Factor VIIa (NovoSeven)": {
        "patterns": [r"coagulation factor viia"],
        "match_column": "Order Name",
        "target_unit": "mg",
        "unit_conversions": {
            "mg":  1.0,
            "mcg": 0.001,   # 1 mcg = 0.001 mg
        },
        "dose_from_name_pattern": r"([\d,]+)\s*(mcg|mg)\s*$",
        "default_dose": {"amount": None, "unit": "mg"},  # TODO: fill in
    },
    "Coagulation Factor IX (Benefix)": {
        "patterns": [r"coagulation factor ix"],
        "match_column": "Order Name",
        "target_unit": "Units",
        "unit_conversions": {
            "Units": 1.0,
        },
        "dose_from_name_pattern": r"([\d,]+)\s*(Units)\s*$",
        "default_dose": {"amount": None, "unit": "Units"},  # TODO: fill in
    },
}

## Core Functions

In [ ]:
def classify_row(row, schema):
    """Return the schema key matching this row, or None if no pattern matches."""
    for drug_type, config in schema.items():
        col = config.get("match_column", "Order Name")
        val = str(row.get(col) or "")
        if any(re.search(p, val, re.IGNORECASE) for p in config["patterns"]):
            return drug_type
    return None


def apply_schema(df, schema):
    """Add a 'drug_type' column classifying each row according to the schema."""
    df = df.copy()
    df["drug_type"] = df.apply(lambda row: classify_row(row, schema), axis=1)
    return df


def _convert_dose(amount, unit, config):
    """Convert (amount, unit) to target_unit using config's unit_conversions.
    Returns NaN if unit is not in unit_conversions or amount is NaN."""
    factor = config["unit_conversions"].get(unit)
    if factor is None or pd.isna(amount):
        return float("nan")
    return amount * factor


def _resolve_ordered_dose(row, config):
    """Resolve the dose for a row using a three-step fallback chain:

    1. Ordered Dose Amount + Ordered Dose Unit  (if unit is in unit_conversions)
    2. Amount + unit extracted from Order Name  (via dose_from_name_pattern)
    3. default_dose["amount"] + default_dose["unit"]  (if amount is not None)

    Returns (amount: float, unit: str) — both None/nan if nothing resolves.
    """
    conversions = config["unit_conversions"]

    # 1. Ordered Dose Amount
    ordered_amount = row.get("Ordered Dose Amount")
    ordered_unit = str(row.get("Ordered Dose Unit") or "")
    if not pd.isna(ordered_amount) and ordered_unit in conversions:
        return float(ordered_amount), ordered_unit

    # 2. Dose from Order Name
    pattern = config.get("dose_from_name_pattern")
    if pattern:
        order_name = str(row.get("Order Name") or "")
        m = re.search(pattern, order_name, re.IGNORECASE)
        if m:
            try:
                return float(m.group(1).replace(",", "")), m.group(2)
            except (ValueError, IndexError):
                pass

    # 3. Default dose
    default = config.get("default_dose", {})
    if default.get("amount") is not None:
        return float(default["amount"]), default["unit"]

    return float("nan"), None


def get_dose_counts(df, schema):
    """Return a Series of dose counts per drug type (schema keys only)."""
    return pd.Series(
        {drug_type: int((df["drug_type"] == drug_type).sum()) for drug_type in schema},
        name="dose_count",
    )


def get_total_amounts(df, schema):
    """Return a DataFrame with total ordered dose per drug type.

    Dose per row is resolved via _resolve_ordered_dose (ordered amount →
    name extraction → default_dose), then converted to target_unit.

    Columns:
        total_amount    – sum in target_unit
        target_unit     – unit total_amount is expressed in
        dose_count      – rows matched to this drug type
        excluded_doses  – rows where dose could not be resolved/converted
    """
    rows = []
    for drug_type, config in schema.items():
        subset = df[df["drug_type"] == drug_type].copy()

        def resolve_and_convert(row, cfg=config):
            amount, unit = _resolve_ordered_dose(row, cfg)
            return _convert_dose(amount, unit, cfg)

        subset["_converted"] = subset.apply(resolve_and_convert, axis=1)
        rows.append({
            "drug_type":      drug_type,
            "target_unit":    config["target_unit"],
            "total_amount":   subset["_converted"].sum(),
            "dose_count":     len(subset),
            "excluded_doses": int(subset["_converted"].isna().sum()),
        })
    return pd.DataFrame(rows).set_index("drug_type")

## Results

In [6]:
df_classified = apply_schema(df, DRUG_SCHEMA)

# Rows that didn't match any schema entry
unmatched = df_classified[df_classified["drug_type"].isna()]
if not unmatched.empty:
    print(f"⚠  {len(unmatched)} unmatched rows:")
    print(unmatched[["Order Name", "Ordered Dose Amount", "Ordered Dose Unit"]].value_counts().to_string())
    print()

# ── Dose counts ──────────────────────────────────────────────────────────────
counts = get_dose_counts(df_classified, DRUG_SCHEMA)
print("Doses per drug type:")
print(counts[counts > 0].to_string())

# ── Total amounts ─────────────────────────────────────────────────────────────
print("\nTotal ordered amounts:")
amounts = get_total_amounts(df_classified, DRUG_SCHEMA)
print(amounts[amounts["dose_count"] > 0].to_string())

Doses per drug type:
Albumin 5%                                 1575
Albumin 25%                                 932
Balfaxar (4F-PCC)                           121
Kcentra (4F-PCC)                              2
Antithrombin III (Thrombate)                  4
Antihemophilic Factor-VWF (Alphanate)        36
Antihemophilic Factor rAHF-PFM (Advate)      29
Coagulation Factor VIIa (NovoSeven)           3

Total ordered amounts:
                                        target_unit  total_amount  dose_count  excluded_doses
drug_type                                                                                    
Albumin 5%                                        g      20286.55        1575               0
Albumin 25%                                       g      16132.25         932               0
Balfaxar (4F-PCC)                             Units     209000.00         121               0
Kcentra (4F-PCC)                              Units          0.00           2               2
Antithr